# Module 2: Banking Transformations

**Objective**: Learn practical data transformations for banking analytics.

## What You'll Learn
1. Multi-table joins (Customers ↔ Accounts ↔ Transactions)
2. Customer segmentation logic
3. Aggregations and grouping
4. Creating derived columns with business logic

In [1]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module02").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

✅ Databricks Connect | Spark 4.1.0
Mode: databricks


In [2]:
# ── Load data (S3 Parquet or local CSV) ──
customers_df = spark.read.parquet(f"{S3_RAW}/customers") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "customers.csv"), header=True, inferSchema=True)
accounts_df = spark.read.parquet(f"{S3_RAW}/accounts") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)
transactions_df = spark.read.format("delta").load(f"{S3_RAW}/transactions_delta") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)
branches_df = spark.read.parquet(f"{S3_RAW}/branches") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "branches.csv"), header=True, inferSchema=True)

print(f"Loaded: {customers_df.count()} customers, {accounts_df.count()} accounts, {transactions_df.count()} transactions")

Loaded: 10000 customers, 16442 accounts, 5000000 transactions


## Multi-Table Joins

In [3]:
# Join customers with accounts
customer_accounts = customers_df.join(accounts_df, "customer_id", "inner")
customer_accounts.select("customer_id", "name", "segment", "account_id", "balance").show(5)

+-----------+--------------+-------+----------+-------------+
|customer_id|          name|segment|account_id|      balance|
+-----------+--------------+-------+----------+-------------+
| CUST007475|Customer 07475|   Mass|ACCT012332|   8398262.39|
| CUST007475|Customer 07475|   Mass|ACCT012333|1.212603567E7|
| CUST007476|Customer 07476|   Mass|ACCT012334|1.889268539E7|
| CUST007477|Customer 07477|   Mass|ACCT012335|   2755330.01|
| CUST007478|Customer 07478|   Mass|ACCT012336|   1020710.81|
+-----------+--------------+-------+----------+-------------+
only showing top 5 rows


## Customer Aggregations

In [4]:
from pyspark.sql.functions import count, sum, avg, round as spark_round, col, when, date_format, to_date

In [5]:
# Customer summary
customer_summary = accounts_df.groupBy("customer_id").agg(
    count("account_id").alias("account_count"),
    spark_round(sum("balance"), 2).alias("total_balance"),
    spark_round(avg("balance"), 2).alias("avg_balance")
)
customer_summary.orderBy(col("total_balance").desc()).show(10)

+-----------+-------------+------------------+-----------------+
|customer_id|account_count|     total_balance|      avg_balance|
+-----------+-------------+------------------+-----------------+
| CUST004750|            3|1.2462703053777E11|4.154234351259E10|
| CUST000621|            3|1.1948662840498E11|3.982887613499E10|
| CUST008150|            3| 1.153007508303E11| 3.84335836101E10|
| CUST002603|            3|1.1480851364991E11|3.826950454997E10|
| CUST009481|            3|1.0858819564595E11|3.619606521532E10|
| CUST001775|            3|1.0082519449151E11|3.360839816384E10|
| CUST005044|            3|1.0004758086192E11|3.334919362064E10|
| CUST007747|            3| 9.955017590227E10|3.318339196742E10|
| CUST004892|            2| 9.815940347914E10|4.907970173957E10|
| CUST004656|            2|  9.71458185112E10| 4.85729092556E10|
+-----------+-------------+------------------+-----------------+
only showing top 10 rows


## Customer Segmentation Logic

In [6]:
# Re-segment based on balance
customer_profile = customers_df.join(customer_summary, "customer_id", "left").fillna({"total_balance": 0})
customer_resegment = customer_profile.withColumn(
    "calculated_segment",
    when(col("total_balance") >= 5_000_000_000, "UHNW")
    .when(col("total_balance") >= 500_000_000, "HNW")
    .when(col("total_balance") >= 100_000_000, "Affluent")
    .when(col("total_balance") >= 20_000_000, "Mass Affluent")
    .otherwise("Mass")
)
customer_resegment.select("name", "segment", "calculated_segment", "total_balance").show(10)

+--------------+-------------+------------------+---------------+
|          name|      segment|calculated_segment|  total_balance|
+--------------+-------------+------------------+---------------+
|Customer 06251|         Mass|              Mass|  1.063612138E7|
|Customer 06252|Mass Affluent|          Affluent| 1.3522805773E8|
|Customer 06253|         Mass|              Mass|  1.811233082E7|
|Customer 06254|Mass Affluent|          Affluent| 1.0016471999E8|
|Customer 06255|          HNW|               HNW|2.99095530645E9|
|Customer 06256|         Mass|              Mass|  1.514097044E7|
|Customer 06257|Mass Affluent|          Affluent| 1.3316701797E8|
|Customer 06258|Mass Affluent|     Mass Affluent|  5.713734625E7|
|Customer 06259|Mass Affluent|     Mass Affluent|  8.510116382E7|
|Customer 06260|         Mass|     Mass Affluent|  2.592709984E7|
+--------------+-------------+------------------+---------------+
only showing top 10 rows


## Transaction Analysis

In [7]:
# Monthly by channel
txn_parsed = transactions_df.withColumn("txn_month", date_format(to_date(col("txn_datetime")), "yyyy-MM"))
monthly_by_channel = txn_parsed.groupBy("txn_month", "channel").agg(count("*").alias("count"), sum("amount").alias("total"))
monthly_by_channel.orderBy("txn_month").show(15)

+---------+----------------+-----+--------------------+
|txn_month|         channel|count|               total|
+---------+----------------+-----+--------------------+
|  2025-01|             API|70655|3.917239225343873E12|
|  2025-01|Internet Banking|71024| 3.96100423173662E12|
|  2025-01|          Branch|70992|3.933845600689784...|
|  2025-01|             POS|70664|3.928462615533408E12|
|  2025-01|      Mobile App|70819|3.924665539435593E12|
|  2025-01|             ATM|70404|3.917175605840881...|
|  2025-02|Internet Banking|64150|3.530528354875180...|
|  2025-02|          Branch|64031|3.567835897754383...|
|  2025-02|             POS|63632|3.555105351798108E12|
|  2025-02|             ATM|64272|3.591315417100188E12|
|  2025-02|      Mobile App|63813|3.537048210491230...|
|  2025-02|             API|63872|3.550060993394684E12|
|  2025-03|             POS|70640|3.944063533545708...|
|  2025-03|Internet Banking|70740| 3.91305075831661E12|
|  2025-03|          Branch|70976|3.919087300167

In [8]:
# Transaction categorization
txn_cat = transactions_df.withColumn("size",
    when(col("amount") >= 100_000_000, "Large").when(col("amount") >= 10_000_000, "Medium").otherwise("Small")
)
txn_cat.groupBy("size").count().show()

+------+-------+
|  size|  count|
+------+-------+
| Large| 398369|
|Medium|3397519|
| Small|1204112|
+------+-------+



## Practice Exercises
1. Find top 10 branches by transaction volume
2. Calculate deposit-to-withdrawal ratio per customer
3. Find customers with no transactions (left_anti join)

In [9]:
branches_df.printSchema()
transactions_df.printSchema()

root
 |-- branch_id: string (nullable = true)
 |-- branch_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- address: string (nullable = true)
 |-- opened_date: string (nullable = true)

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_datetime: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- reference: string (nullable = true)
 |-- description: string (nullable = true)
 |-- txn_date: date (nullable = true)



In [10]:
def generate_cols_class(df, class_name: str):
    """Generate a column-name constants class from a DataFrame's schema."""
    lines = [f"class {class_name}:"]
    for field in df.schema.fields:
        const_name = field.name.upper()
        lines.append(f'    {const_name} = "{field.name}"')
    print("\n".join(lines))

In [11]:
generate_cols_class(branches_df, "BranchCols")
generate_cols_class(transactions_df, "TransactionCols")
generate_cols_class(customers_df,"CustomerCol")
generate_cols_class(accounts_df,"AccountCol")

class BranchCols:
    BRANCH_ID = "branch_id"
    BRANCH_NAME = "branch_name"
    REGION = "region"
    CITY = "city"
    ADDRESS = "address"
    OPENED_DATE = "opened_date"
class TransactionCols:
    TXN_ID = "txn_id"
    ACCOUNT_ID = "account_id"
    TXN_DATETIME = "txn_datetime"
    TXN_TYPE = "txn_type"
    AMOUNT = "amount"
    CURRENCY = "currency"
    CHANNEL = "channel"
    MERCHANT_CATEGORY = "merchant_category"
    STATUS = "status"
    REFERENCE = "reference"
    DESCRIPTION = "description"
    TXN_DATE = "txn_date"
class CustomerCol:
    CUSTOMER_ID = "customer_id"
    NAME = "name"
    EMAIL = "email"
    PHONE = "phone"
    SEGMENT = "segment"
    REGISTRATION_DATE = "registration_date"
    KYC_STATUS = "kyc_status"
    DATE_OF_BIRTH = "date_of_birth"
    GENDER = "gender"
    NATIONALITY = "nationality"
class AccountCol:
    ACCOUNT_ID = "account_id"
    CUSTOMER_ID = "customer_id"
    BRANCH_ID = "branch_id"
    ACCOUNT_TYPE = "account_type"
    BALANCE = "balance"
    

In [17]:
class BranchCols:
    BRANCH_ID = "branch_id"
    BRANCH_NAME = "branch_name"
    REGION = "region"
    CITY = "city"
    ADDRESS = "address"
    OPENED_DATE = "opened_date"
class TransactionCols:
    TXN_ID = "txn_id"
    ACCOUNT_ID = "account_id"
    TXN_DATETIME = "txn_datetime"
    TXN_TYPE = "txn_type"
    AMOUNT = "amount"
    CURRENCY = "currency"
    CHANNEL = "channel"
    MERCHANT_CATEGORY = "merchant_category"
    STATUS = "status"
    REFERENCE = "reference"
    DESCRIPTION = "description"
    TXN_DATE = "txn_date"
class CustomerCols:
    CUSTOMER_ID = "customer_id"
    NAME = "name"
    EMAIL = "email"
    PHONE = "phone"
    SEGMENT = "segment"
    REGISTRATION_DATE = "registration_date"
    KYC_STATUS = "kyc_status"
    DATE_OF_BIRTH = "date_of_birth"
    GENDER = "gender"
    NATIONALITY = "nationality"
class AccountCols:
    ACCOUNT_ID = "account_id"
    CUSTOMER_ID = "customer_id"
    BRANCH_ID = "branch_id"
    ACCOUNT_TYPE = "account_type"
    BALANCE = "balance"
    CURRENCY = "currency"
    STATUS = "status"
    OPENED_DATE = "opened_date"
    LAST_ACTIVITY_DATE = "last_activity_date"

In [ ]:
from pyspark.sql import functions as F

# If transactions links to branches through an accounts table:
top_10_branches = \
    transactions_df \
    .join(accounts_df, transactions_df[TransactionCols.ACCOUNT_ID] == accounts_df[AccountCols.ACCOUNT_ID]) \
    .join(customers_df, accounts_df[AccountCols.CUSTOMER_ID] == customers_df[CustomerCols.CUSTOMER_ID]) \
    .groupBy(BranchCols.BRANCH_ID) \
    .count() \
    .orderBy(F.col("count").desc()) \
    # .limit(10)

top_10_branches.show()

+---------+------+
|branch_id| count|
+---------+------+
| BR000036|116587|
| BR000016| 91784|
| BR000040| 91331|
| BR000008| 87827|
| BR000024| 83921|
| BR000079| 81695|
| BR000030| 81628|
| BR000026| 80872|
| BR000046| 79998|
| BR000012| 79069|
| BR000084| 78030|
| BR000055| 76594|
| BR000058| 75302|
| BR000022| 72529|
| BR000009| 70933|
| BR000054| 68188|
| BR000004| 67807|
| BR000061| 67533|
| BR000074| 67273|
| BR000082| 67217|
+---------+------+
only showing top 20 rows


In [22]:
from pyspark.sql.functions import col, sum, when, round as spark_round

# Join transactions → accounts to get customer_id
txn_with_customer = transactions_df.join(accounts_df, "account_id") \
    .select("customer_id", "txn_type", "amount")

# Deposit-to-withdrawal ratio per customer
deposit_withdrawal_ratio = txn_with_customer.groupBy("customer_id").agg(
    spark_round(sum(when(col("txn_type") == "Deposit", col("amount")).otherwise(0)), 2).alias("total_deposits"),
    spark_round(sum(when(col("txn_type") == "Withdrawal", col("amount")).otherwise(0)), 2).alias("total_withdrawals"),
).withColumn(
    "deposit_to_withdrawal_ratio",
    spark_round(
        when(col("total_withdrawals") > 0, col("total_deposits") / col("total_withdrawals"))
        .otherwise(None),
        4
    )
)

deposit_withdrawal_ratio.orderBy(col("deposit_to_withdrawal_ratio").desc()).show(15)

+-----------+--------------+-----------------+---------------------------+
|customer_id|total_deposits|total_withdrawals|deposit_to_withdrawal_ratio|
+-----------+--------------+-----------------+---------------------------+
| CUST003686| 4.022934173E7|        802226.34|                    50.1471|
| CUST006218|    5203726.86|        115761.72|                    44.9521|
| CUST000622|    8070619.22|        250311.85|                    32.2423|
| CUST004273|1.1907895074E8|       3713822.75|                    32.0637|
| CUST005251|    3184678.41|        118963.52|                    26.7702|
| CUST004167|    5602851.15|        270409.11|                    20.7199|
| CUST000268|  9.42292355E7|       4936639.03|                    19.0877|
| CUST008775|    3998142.98|        212925.67|                    18.7772|
| CUST007211|    5379492.57|        305593.43|                    17.6034|
| CUST002912|    2850541.74|        168938.86|                    16.8732|
| CUST004594| 6.210783552

In [23]:
# First, get all customer_ids that have at least one transaction
# (transactions link to customers through accounts)
accounts_with_txns = accounts_df.join(transactions_df, "account_id") \
    .select("customer_id").distinct()
# left_anti: keep only customers NOT found in accounts_with_txns
customers_no_txns = customers_df.join(accounts_with_txns, "customer_id", "left_anti")
print(f"Customers with no transactions: {customers_no_txns.count()}")
customers_no_txns.select("customer_id", "name", "segment", "registration_date").show(15)

Customers with no transactions: 481
+-----------+--------------+-------+-----------------+
|customer_id|          name|segment|registration_date|
+-----------+--------------+-------+-----------------+
| CUST006955|Customer 06955|   Mass|       2023-05-21|
| CUST006801|Customer 06801|   Mass|       2017-08-03|
| CUST006642|Customer 06642|   Mass|       2023-06-13|
| CUST007162|Customer 07162|   Mass|       2016-09-26|
| CUST007050|Customer 07050|   Mass|       2020-03-05|
| CUST007440|Customer 07440|   Mass|       2022-05-25|
| CUST006320|Customer 06320|   Mass|       2023-09-09|
| CUST007289|Customer 07289|   Mass|       2024-01-09|
| CUST007148|Customer 07148|   Mass|       2023-12-25|
| CUST006424|Customer 06424|   Mass|       2017-05-06|
| CUST006287|Customer 06287|   Mass|       2021-12-02|
| CUST006783|Customer 06783|   Mass|       2023-11-02|
| CUST006449|Customer 06449|   Mass|       2024-02-28|
| CUST006733|Customer 06733|   Mass|       2020-06-15|
| CUST006329|Customer 06329| 

In [ ]:
spark.stop()